In [ ]:
import torch
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm
import os

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

In [ ]:
from tablevault import tablevault

vault = tablevault.Vault(user_id="jinjin",
                            process_name="hf_pipeline_sequence_classification_mrpc",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [ ]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding

In [ ]:
model_name = "textattack/distilbert-base-uncased-MRPC"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()

clf = pipeline(
    task="text-classification",
    model=model,
    tokenizer=tokenizer,
)

id2label = {int(k): v for k, v in model.config.id2label.items()}
label2id = {str(k): int(v) for k, v in model.config.label2id.items()}

print(model_name)
print(id2label)
print("pipeline device:", clf.device)

In [ ]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_dict(ds)
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

print("num_examples:", len(y_true))
print("positive_rate:", y_true.mean())

In [ ]:
inputs = [{"text": s1, "text_pair": s2} for s1, s2 in zip(sent1, sent2)]

batch_size = 64
preds = []
scores = []
vault.create_record_list("mrpc_score_predictions", column_names=["prediction", "score"])

with torch.no_grad():
    for i in tqdm(range(0, len(inputs), batch_size)):
        batch_inputs = inputs[i:i + batch_size]
        outputs = clf(
            batch_inputs,
            batch_size=batch_size,
            truncation=True,
            max_length=128,
            function_to_apply="softmax",
        )

        for j, out in enumerate(outputs):
            label = out["label"]
            pred = label2id[label] if label in label2id else int(str(label).split("_")[-1])
            preds.append(pred)
            score = float(out["score"])
            scores.append(score)

            vault.append_record("mrpc_score_predictions", {"prediction": pred, "score": score}, 
                           input_items = {"glue_mrpc_validation": [i + j, i + j + 1]}
                           )

y_pred = np.array(preds)
y_score = np.array(scores)
print("done")

In [ ]:
description = "INSERT TEXT HERE ABOUT mrpc_score_predictions"
embedding = get_embeddings(description)
vault.create_description("mrpc_score_predictions", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("mrpc_score_predictions", cat, embedding, prop)

In [ ]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"])

print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))

In [ ]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "label:", id2label[int(y_pred[i])], "score:", float(y_score[i]))

In [ ]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "score:", float(y_score[i]))

In [ ]:
vault.create_record_list("hf_pipeline_sequence_classification_mrpc_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("hf_pipeline_sequence_classification_mrpc_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "mrpc_score_predictions": [0, len(ds)]
                    })

summary

description = "INSERT TEXT HERE ABOUT hf_pipeline_sequence_classification_mrpc_summary"
embedding = get_embeddings(description)
vault.create_description("hf_pipeline_sequence_classification_mrpc_summary", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("hf_pipeline_sequence_classification_mrpc_summary", cat, embedding, prop)

In [ ]:
description = "INSERT TEXT HERE ABOUT hf_pipeline_sequence_classification_mrpc process" # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("hf_pipeline_sequence_classification_mrpc", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("hf_pipeline_sequence_classification_mrpc", cat, embedding, prop)